# M3-B1 — Mission étoile : préparer la 3ᵉ source pour du RAG — SUJET

> **Bonus optionnel, non noté.** Acerox t'a transmis un corpus de **rapports
> techniques** (`rapports_techniques_2024.zip`, 5 fichiers `.md`). C'est du
> **texte non structuré** : impossible à ranger dans une table SQL. Tu vas le
> **préparer** pour une exploitation par similarité sémantique.

## 🚫 Garde-fou
> **Préparation seulement** : tu t'arrêtes à la **récupération** (retrieval).
> La génération augmentée par un LLM, c'est **M7-B2**. Pour la prédiction de
> défauts (tabulaire), un RAG **n'aide pas** : ce corpus sert l'**aide au
> diagnostic humain**. Embeddings **locaux** (aucune clé API).

## Pré-requis
```bash
# Dézippe rapports_techniques_2024.zip dans  ../data/rapports_techniques_2024/
# Décommente le bloc bonus de requirements.txt puis :
pip install chromadb sentence-transformers
```

## 1. Repérer les rapports (fourni)

In [1]:
from pathlib import Path

RAPPORTS = Path("../data/rapports_techniques_2024")
EMBED_MODEL = "all-MiniLM-L6-v2"  # Adapté au corpus ? (regarde la langue des rapports)

fichiers = sorted(RAPPORTS.glob("*.md"))
print(f"{len(fichiers)} rapports trouvés :")
for f in fichiers:
    print(" -", f.name)

5 rapports trouvés :
 - RT-2026-001_Roubaix_ligne1.md
 - RT-2026-002_Lyon_ligne2.md
 - RT-2026-003_Lyon_ligne4.md
 - RT-2026-004_Lyon_ligne4.md
 - RT-2026-005_Lyon_ligne2.md


## 2. DocumentLoader (fait main) 

Écris une fonction qui lit un `.md` et renvoie un dict avec au moins :
`source`, `reference`, `date` (extraits de l'en-tête par regex) et `texte`.
Pas besoin de LangChain.

In [13]:
import re


def charger_rapport(path: Path) -> dict:
    contenu = path.read_text(encoding="utf-8")

    # Extrait les métadonnées dans l'en-tête du document Markdown.
    ref_match = re.search(r"^\s*(?:reference|référence)\s*:\s*(.+)$", contenu, re.IGNORECASE | re.MULTILINE)
    date_match = re.search(r"^\s*date\s*:\s*(.+)$", contenu, re.IGNORECASE | re.MULTILINE)

    return {
        "source": path.name,
        "reference": ref_match.group(1).strip() if ref_match else "inconnue",
        "date": date_match.group(1).strip() if date_match else "inconnue",
        "texte": contenu.strip(),
    }


docs = [charger_rapport(f) for f in fichiers]

## 3. Segmentation (chunking) 

Implémente **2 stratégies** et **justifie ton choix** :
- **par titre Markdown** (`##`) : chunks cohérents avec la structure ;
- **par taille fixe** (avec recouvrement) : chunks homogènes.

> C'est l'item syllabus « implémentation de techniques de segmentation ».

In [47]:
def extraire_valeur(pattern_cle: str, texte: str, defaut: str = "inconnue") -> str:
    # Capture une valeur de type "Cle : valeur" meme dans une ligne citationnee avec "|".
    motif = rf"(?:^|[|>\n\r])\s*(?:{pattern_cle})\s*:\s*([^|\n\r]+)"
    match = re.search(motif, texte, flags=re.IGNORECASE)
    return match.group(1).strip() if match else defaut


def extraire_premier_h2(texte: str) -> str:
    match = re.search(r"^\s*##\s+(.+?)\s*$", texte, flags=re.MULTILINE)
    return match.group(1).strip() if match else "incident_inconnu"


def chunk_par_titre(doc: dict) -> list[dict]:
    texte_doc = doc.get("texte", "")
    if not texte_doc.strip():
        return []

    source = doc.get("source", "inconnue")
    incident = extraire_premier_h2(texte_doc)
    reference = extraire_valeur(r"reference|référence", texte_doc, doc.get("reference", "inconnue"))
    date = extraire_valeur(r"date", texte_doc, doc.get("date", "inconnue"))
    site = extraire_valeur(r"site", texte_doc, "site_inconnu")
    ligne = extraire_valeur(r"ligne", texte_doc, "ligne_inconnue")
    equipement = extraire_valeur(r"equipement|équipement", texte_doc, "equipement_inconnu")

    # 1 chunk par rapport: le texte stocke le nom de l'incident (1er H2).
    return [
        {
            "id": f"{source}::titre::0",
            "texte": incident,
            "meta": {
                "source": source,
                "reference": reference,
                "date": date,
                "site": site,
                "ligne": ligne,
                "equipement": equipement,
            },
        }
    ]


def chunk_taille_fixe(doc: dict, taille: int = 500, overlap: int = 80) -> list[dict]:
    texte = doc.get("texte", "").strip()
    if not texte:
        return []

    if taille <= 0:
        raise ValueError("taille doit etre > 0")
    if overlap < 0:
        raise ValueError("overlap doit etre >= 0")
    if overlap >= taille:
        raise ValueError("overlap doit etre strictement inferieur a taille")

    source = doc.get("source", "inconnue")
    reference = extraire_valeur(r"reference|référence", texte, doc.get("reference", "inconnue"))
    date = extraire_valeur(r"date", texte, doc.get("date", "inconnue"))
    site = extraire_valeur(r"site", texte, "site_inconnu")
    ligne = extraire_valeur(r"ligne", texte, "ligne_inconnue")
    equipement = extraire_valeur(r"equipement|équipement", texte, "equipement_inconnu")

    pas = taille - overlap
    chunks: list[dict] = []

    for i, debut in enumerate(range(0, len(texte), pas)):
        fin = debut + taille
        contenu = texte[debut:fin].strip()
        if not contenu:
            continue
        chunks.append(
            {
                "id": f"{source}::fixe::{i}",
                "texte": contenu,
                "meta": {
                    "source": source,
                    "reference": reference,
                    "date": date,
                    "site": site,
                    "ligne": ligne,
                    "equipement": equipement,
                },
            }
        )

    return chunks


# Compare le nombre de chunks des 2 strategies et garde-en une pour la suite.
docs = [charger_rapport(f) for f in fichiers]
chunks_titre = [c for d in docs for c in chunk_par_titre(d)]
chunks_fixes = [c for d in docs for c in chunk_taille_fixe(d)]
print("Chunks par titre:", len(chunks_titre))
print("Chunks taille fixe:", len(chunks_fixes))

# Apercu de controle: on doit obtenir 1 incident par fichier avec les metas attendues.
for c in chunks_titre:
    print(" -", c["id"], "|", c["texte"], "|", c["meta"])

# On conserve la strategie par titre pour la suite, car elle renvoie 1 incident/rapport.

Chunks par titre: 5
Chunks taille fixe: 10
 - RT-2026-001_Roubaix_ligne1.md::titre::0 | Dérive thermique ligne 1 | {'source': 'RT-2026-001_Roubaix_ligne1.md', 'reference': 'RT-2026-001', 'date': '2026-04-01', 'site': 'Roubaix', 'ligne': '1', 'equipement': 'capteur de température S12'}
 - RT-2026-002_Lyon_ligne2.md::titre::0 | Vibrations anormales ligne 2 | {'source': 'RT-2026-002_Lyon_ligne2.md', 'reference': 'RT-2026-002', 'date': '2026-04-08', 'site': 'Lyon', 'ligne': '2', 'equipement': 'capteur de vibration S09'}
 - RT-2026-003_Lyon_ligne4.md::titre::0 | Baisse de débit ligne 4 | {'source': 'RT-2026-003_Lyon_ligne4.md', 'reference': 'RT-2026-003', 'date': '2026-04-11', 'site': 'Lyon', 'ligne': '4', 'equipement': 'débitmètre S05'}
 - RT-2026-004_Lyon_ligne4.md::titre::0 | Défaut qualité lot L6925 | {'source': 'RT-2026-004_Lyon_ligne4.md', 'reference': 'RT-2026-004', 'date': '2026-04-18', 'site': 'Lyon', 'ligne': '4', 'equipement': 'contrôle dimensionnel'}
 - RT-2026-005_Lyon_ligne2.m

## 4. Embeddings + indexation ChromaDB — **TODO**

Encode chaque chunk avec `SentenceTransformer(EMBED_MODEL)` (local), puis range
les chunks dans une collection ChromaDB **persistante**.

> Pense à **vider** la collection si elle existe déjà (idempotence du notebook).

In [48]:
import chromadb
from sentence_transformers import SentenceTransformer


chunks_selectionnes = chunks_titre

ids = [chunk["id"] for chunk in chunks_selectionnes]
documents = [chunk["texte"] for chunk in chunks_selectionnes]
metadatas = [chunk["meta"] for chunk in chunks_selectionnes]

# Controle attendu: 5 rapports => 5 incidents indexes.
print(f"Incidents prepares: {len(documents)}")
for i, (doc_txt, meta) in enumerate(zip(documents, metadatas), start=1):
    print(f"{i}. {doc_txt} | source={meta.get('source')} | site={meta.get('site')} | ligne={meta.get('ligne')} | equipement={meta.get('equipement')}")

# Encoder localement
modele = SentenceTransformer(EMBED_MODEL)
embeddings = modele.encode(documents).tolist()

# Indexation persistante et idempotente
client = chromadb.PersistentClient(path="./chroma_acerox")
collection_name = "rapports_acerox"

try:
    client.delete_collection(collection_name)
except Exception:
    pass

col = client.create_collection(name=collection_name)
col.add(ids=ids, documents=documents, embeddings=embeddings, metadatas=metadatas)

print(f"Collection '{collection_name}' indexee avec {len(ids)} incidents.")

Incidents prepares: 5
1. Dérive thermique ligne 1 | source=RT-2026-001_Roubaix_ligne1.md | site=Roubaix | ligne=1 | equipement=capteur de température S12
2. Vibrations anormales ligne 2 | source=RT-2026-002_Lyon_ligne2.md | site=Lyon | ligne=2 | equipement=capteur de vibration S09
3. Baisse de débit ligne 4 | source=RT-2026-003_Lyon_ligne4.md | site=Lyon | ligne=4 | equipement=débitmètre S05
4. Défaut qualité lot L6925 | source=RT-2026-004_Lyon_ligne4.md | site=Lyon | ligne=4 | equipement=contrôle dimensionnel
5. Incident capteur S02 | source=RT-2026-005_Lyon_ligne2.md | site=Lyon | ligne=2 | equipement=supervision IoT


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given


Collection 'rapports_acerox' indexee avec 5 incidents.


## 5. Interroger par similarité

In [50]:
def interroger(question: str, k: int = 3) -> None:
    if not question or not question.strip():
        raise ValueError("La question ne doit pas etre vide.")
    if k <= 0:
        raise ValueError("k doit etre > 0.")

    question_embedding = modele.encode([question]).tolist()
    resultats = col.query(
        query_embeddings=question_embedding,
        n_results=k,
        include=["documents", "metadatas", "distances"],
    )

    documents_res = resultats.get("documents", [[]])[0]
    metadatas_res = resultats.get("metadatas", [[]])[0]
    distances_res = resultats.get("distances", [[]])[0]

    print(f"Question : {question}")
    print(f"Top {len(documents_res)} resultats:\n")

    for i, (doc_txt, meta, dist) in enumerate(
        zip(documents_res, metadatas_res, distances_res), start=1
    ):
        print(f"[{i}] distance={dist:.4f}")
        print(f"incident: {doc_txt}")
        print(
            "meta:",
            f"source={meta.get('source', 'N/A')}",
            f"reference={meta.get('reference', 'N/A')}",
            f"date={meta.get('date', 'N/A')}",
            f"site={meta.get('site', 'N/A')}",
            f"ligne={meta.get('ligne', 'N/A')}",
            f"equipement={meta.get('equipement', 'N/A')}",
        )
        print("-" * 80)


# Exemples de questions metier a tester :
interroger("Pourquoi des défauts sur la ligne de laminage ?")
interroger("Quels capteurs sont peu fiables ?")
interroger("Risque RGPD avec les données opérateurs de l'ERP ?")

Question : Pourquoi des défauts sur la ligne de laminage ?
Top 3 resultats:

[1] distance=0.9387
incident: Baisse de débit ligne 4
meta: source=RT-2026-003_Lyon_ligne4.md reference=RT-2026-003 date=2026-04-11 site=Lyon ligne=4 equipement=débitmètre S05
--------------------------------------------------------------------------------
[2] distance=1.3231
incident: Vibrations anormales ligne 2
meta: source=RT-2026-002_Lyon_ligne2.md reference=RT-2026-002 date=2026-04-08 site=Lyon ligne=2 equipement=capteur de vibration S09
--------------------------------------------------------------------------------
[3] distance=1.4477
incident: Dérive thermique ligne 1
meta: source=RT-2026-001_Roubaix_ligne1.md reference=RT-2026-001 date=2026-04-01 site=Roubaix ligne=1 equipement=capteur de température S12
--------------------------------------------------------------------------------
Question : Quels capteurs sont peu fiables ?
Top 3 resultats:

[1] distance=1.0061
incident: Incident capteur S02
meta

## ✅ Vérification (checklist)

- [x] J'ai chargé les 5 rapports + leurs métadonnées (loader fait main)
- [x] J'ai comparé 2 stratégies de **segmentation** et justifié mon choix
- [x] J'ai indexé les chunks dans ChromaDB avec des embeddings **locaux**
- [x] Une requête en langage naturel récupère les bons passages
- [ ] J'ai vérifié que le **modèle d'embedding est adapté à la langue du corpus** (les rapports sont en français)
- [ ] J'ai écrit 3 lignes (dans ma note) sur **relationnel vs vectoriel**
- [ ] Je sais où s'arrête la **préparation** vs le **RAG complet** (M7-B2)

## ⭐ Pour aller plus loin
- Filtrer par métadonnée (`where={...}`) pour ne chercher que les rapports récents.
- Brancher un LLM local (Ollama) sur les chunks — mais : est-ce justifié ici ?